<a href="https://colab.research.google.com/github/kumarsirish/-RAG_legal_docs/blob/main/RAG_Assg_Legal_Documents_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Extracting Information from Legal Documents Using RAG**

## **Objective**

The main objective of this assignment is to process and analyse a collection text files containing legal agreements (e.g., NDAs) to prepare them for implementing a **Retrieval-Augmented Generation (RAG)** system. This involves:

* Understand the Cleaned Data : Gain a comprehensive understanding of the structure, content, and context of the cleaned dataset.
* Perform Exploratory Analysis : Conduct bivariate and multivariate analyses to uncover relationships and trends within the cleaned data.
* Create Visualisations : Develop meaningful visualisations to support the analysis and make findings interpretable.
* Derive Insights and Conclusions : Extract valuable insights from the cleaned data and provide clear, actionable conclusions.
* Document the Process : Provide a detailed description of the data, its attributes, and the steps taken during the analysis for reproducibility and clarity.

The ultimate goal is to transform the raw text data into a clean, structured, and analysable format that can be effectively used to build and train a RAG system for tasks like information retrieval, question-answering, and knowledge extraction related to legal agreements.

### **Business Value**  


The project aims to leverage RAG to enhance legal document processing for businesses, law firms, and regulatory bodies. The key business objectives include:

* Faster Legal Research: <br> Reduce the time lawyers and compliance officers spend searching for relevant case laws, precedents, statutes, or contract clauses.
* Improved Contract Analysis: <br> Automatically extract key terms, obligations, and risks from lengthy contracts.
* Regulatory Compliance Monitoring: <br> Help businesses stay updated with legal and regulatory changes by retrieving relevant legal updates.
* Enhanced Decision-Making: <br> Provide accurate and context-aware legal insights to assist in risk assessment and legal strategy.


**Use Cases**
* Legal Chatbots
* Contract Review Automation
* Tracking Regulatory Changes and Compliance Monitoring
* Case Law Analysis of past judgments
* Due Diligence & Risk Assessment

## **1. Data Loading, Preparation and Analysis** <font color=red> [20 marks] </font><br>

### **1.1 Data Understanding**

The dataset contains legal documents and contracts collected from various sources. The documents are present as text files (`.txt`) in the *corpus* folder.

There are four types of documents in the *courpus* folder, divided into four subfolders.
- `contractnli`: contains various non-disclosure and confidentiality agreements
- `cuad`: contains contracts with annotated legal clauses
- `maud`: contains various merger/acquisition contracts and agreements
- `privacy_qa`: a question-answering dataset containing privacy policies

The dataset also contains evaluation files in JSON format in the *benchmark* folder. The files contain the questions and their answers, along with sources. For the above folders, there is a `json` file: `contractnli.json`, `cuad.json`, `maud.json`. The file structure is as follows:

```
{
    "tests": [
        {
            "query": <question1>,
            "snippets": [{
                    "file_path": <source_file1>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 1>
                },
                {
                    "file_path": <source_file2>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 2>
                }, ....
            ]
        },
        {
            "query": <question2>,
            "snippets": [{<answer context for que 2>}]
        },
        ... <more queries>
    ]
}
```

### **1.2 Load and Preprocess the data** <font color=red> [5 marks] </font><br>

#### Loading libraries

In [ ]:
## The following libraries might be useful
#%pip install -q langchain-openai
#%pip install -U -q langchain-community
#%pip install -U -q langchain-chroma
#%pip install -U -q datasets pyarrow pandas
#%pip install -U -q ragas
#%pip install -U -q rouge_score
#%pip install -q sentence-transformers qdrant-client
#!rm -f requirements.txt
!wget -O requirements.txt https://raw.githubusercontent.com/kumarsirish/-RAG_legal_docs/ollama/requirements.txt
#%pip install -r requirements.txt

In [ ]:
#!pip freeze
# Import essential libraries



#### **1.2.1** <font color=red> [3 marks] </font>
Load all `.txt` files from the folders.

You can utilise document loaders from the options provided by the LangChain community.

Optionally, you can also read the files manually, while ensuring proper handling of encoding issues (e.g., utf-8, latin1). In such case, also store the file content along with metadata (e.g., file name, directory path) for traceability.

In [ ]:
import os
import json
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Download + extract (simple)
!rm -rf corpus benchmarks benchmark corpus.zip benchmark.zip
!wget -q -O corpus.zip https://github.com/kumarsirish/-RAG_legal_docs/raw/main/corpus.zip
!wget -q -O benchmark.zip https://github.com/kumarsirish/-RAG_legal_docs/raw/main/benchmark.zip
!unzip -q -o corpus.zip -d .
!unzip -q -o benchmark.zip -d .

# Some archives unpack as benchmark/; normalize to benchmarks/
if os.path.exists("benchmark") and not os.path.exists("benchmarks"):
    os.rename("benchmark", "benchmarks")

corpus_path = "./corpus"
benchmark_path = "./benchmarks"

# Load corpus .txt files as LangChain Documents
documents = DirectoryLoader(
    corpus_path,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"autodetect_encoding": True},
    silent_errors=True,
).load()

# Optional metadata cleanup
for d in documents:
    d.metadata["file_name"] = os.path.basename(d.metadata.get("source", ""))

# Load benchmark json files
benchmarks = {}
for file_name in os.listdir(benchmark_path):
    if file_name.endswith(".json"):
        file_path = os.path.join(benchmark_path, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            benchmarks[file_name.replace(".json", "")] = json.load(f)

print("Docs loaded:", len(documents))
print("Benchmarks loaded:", list(benchmarks.keys()))


#### **1.2.2** <font color=red> [2 marks] </font>
Preprocess the text data to remove noise and prepare it for analysis.

Remove special characters, extra whitespace, and irrelevant content such as email and telephone contact info.
Normalise text (e.g., convert to lowercase, remove stop words).
Handle missing or corrupted data by logging errors and skipping problematic files.

In [ ]:
# Clean and preprocess the data
# remove emails from the documents
import re
import nltk
nltk.download('stopwords')

def remove_emails(text):
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    return re.sub(email_pattern, '', text)
# remove phone numbers from the documents
def remove_phone_numbers(text):
    phone_pattern = r'\+?\d[\d -]{8,}\d'
    return re.sub(phone_pattern, '', text)
# remove special characters from the documents
def remove_special_characters(text):
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

#remove extra spaces from the documents
def remove_extra_spaces(text):
    return re.sub(r"[ \t]+", " ", text).strip()

#convert the documents to lowercase
def convert_to_lowercase(text):
    return text.lower()

#remove stop words from the documents
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
def remove_stop_words(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

# apply the cleaning functions to the documents
for doc in documents:
    doc.page_content = remove_emails(doc.page_content)
    doc.page_content = remove_phone_numbers(doc.page_content)
    doc.page_content = remove_special_characters(doc.page_content)
    doc.page_content = remove_extra_spaces(doc.page_content)
    doc.page_content = convert_to_lowercase(doc.page_content)
    doc.page_content = remove_stop_words(doc.page_content)
# Check the cleaned document
print(f'Cleaned example document: {documents[0].page_content[:500]}...')






### **1.3 Exploratory Data Analysis** <font color=red> [10 marks] </font><br>

#### **1.3.1** <font color=red> [2 marks] </font>
Calculate the average, maximum and minimum document length.

In [ ]:
# Calculate the average, maximum and minimum document length.
doc_lengths = []  #count of words in each document

print(f'Number of documents: {len(documents)}')
for doc in documents:
    words = doc.page_content.split()
    length = len(words)
    doc_lengths.append(length)


average_length = sum(doc_lengths) / len(documents)
max_length = max(doc_lengths)
min_length = min(doc_lengths)
print(f'Average document length: {average_length}')
print(f'Maximum document length: {max_length}')
print(f'Minimum document length: {min_length}')

#### **1.3.2** <font color=red> [4 marks] </font>
Analyse the frequency of occurence of words and find the most and least occuring words.

Find the 20 most common and least common words in the text. Ignore stop words such as articles and prepositions.

In [ ]:
# Find frequency of occurence of words
from collections import Counter
import pprint
word_freq = Counter()
for doc in documents:
    words = doc.page_content.split()
    #ignore single letter words
    words = [word for word in words if len(word) > 1]
    #ignore prepositions, conjunctions, articles etc. (stop words)
    words = [word for word in words if word not in stop_words]
    #shall and may are very common in legal documents, so ignore them as well
    words = [word for word in words if word not in ['shall', 'may']]
    #ignore roman numerals
    words = [word for word in words if not re.match(r'^[ivxlcdm]+$', word)]
    word_freq.update(words)
# Print the 10 most common words and their frequencies
pprint.pprint(word_freq.most_common(20))



#### **1.3.3** <font color=red> [4 marks] </font>
Analyse the similarity of different documents to each other based on TF-IDF vectors.

Transform some documents to TF-IDF vectors and calculate their similarity matrix using a suitable distance function. If contracts contain duplicate or highly similar clauses, similarity calculation can help detect them.

Identify for the first 10 documents and then for 10 random documents. What do you observe?

In [ ]:
# Transform the page contents of documents - Modularized approach
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import random

def compute_tfidf_vectors(documents_list):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents_list)
    return vectorizer, tfidf_matrix

def compute_pairwise_similarity(tfidf_matrix):
    similarity_matrix = cosine_similarity(tfidf_matrix)
    return similarity_matrix

def analyze_similarity_pairwise(doc_contents, label, doc_indices=None):
    print(f"\n=== {label} ===")
    vectorizer, tfidf_matrix = compute_tfidf_vectors(doc_contents)

    # Compute pairwise similarity matrix
    similarity_matrix = compute_pairwise_similarity(tfidf_matrix)

    # Create labels for rows/columns
    if doc_indices is None:
        labels = [f'Doc {i}' for i in range(len(doc_contents))]
    else:
        labels = [f'Doc {idx}' for idx in doc_indices]

    # Create DataFrame for better visualization
    sim_df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)
    print("\nPairwise Similarity Matrix:")
    print(sim_df.round(4))

    # Print statistics
    # Get upper triangle values (excluding diagonal)
    mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
    upper_triangle = similarity_matrix[mask]

    print(f"\nSimilarity Statistics:")
    print(f"  Average similarity: {upper_triangle.mean():.4f}")
    print(f"  Max similarity: {upper_triangle.max():.4f}")
    print(f"  Min similarity: {upper_triangle.min():.4f}")
    print(f"  Std deviation: {upper_triangle.std():.4f}")

# Analyze first 10 documents (same subdirectory)
doc_contents_first10 = [doc.page_content for doc in documents[:10]]
analyze_similarity_pairwise(doc_contents_first10, "FIRST 10 DOCUMENTS (Same Subdirectory)")

# Analyze 10 random documents (mixed subdirectories)
random_indices = random.sample(range(len(documents)), 10)
doc_contents_random = [documents[idx].page_content for idx in random_indices]
analyze_similarity_pairwise(doc_contents_random, "10 RANDOM DOCUMENTS (Mixed Subdirectories)", random_indices)

In [ ]:
# create a list of 10 random integers



In [ ]:
# Compute similarity scores for 10 random documents



```markdown
# filepath: /home/sirkumar/RAG_legal_docs/RAG_Assg_Legal_Documents_Starter.ipynb
The first 10 documents show higher similarity because they likely contain repeated legal boilerplate, while the random documents are more diverse and have lower off-diagonal similarity.
```

### **1.4 Document Creation and Chunking** <font color=red> [5 marks] </font><br>

#### **1.4.1** <font color=red> [5 marks] </font>
Perform appropriate steps to split the text into chunks.

In [ ]:
# Process files and generate chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = []
for doc in documents:
    doc_chunks = text_splitter.split_text(doc.page_content)
    chunks.extend(doc_chunks)
print(f'Number of chunks generated: {len(chunks)}')
print(f'Example chunk: {chunks[0][:500]}...')

## **2. Vector Database and RAG Chain Creation** <font color=red> [15 marks] </font><br>

### **2.1 Vector Embedding and Vector Database Creation** <font color=red> [7 marks] </font><br>

#### **2.1.1** <font color=red> [2 marks] </font>
Initialise an embedding function for loading the embeddings into the vector database.

Initialise a function to transform the text to vectors using an embedding model. You can also use this function to transform during vector DB creation itself.

In [ ]:

from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

model_name = "all-MiniLM-L6-v2"   
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_function = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": device}
)

print("Loaded:", model_name)
print("Device:", device)


#### Set up Qdrant Host and API Key

To connect to your Qdrant instance, you'll need the `QDRANT_HOST` and `QDRANT_API_KEY`. Please add these to Colab's secrets manager (the 🔑 icon in the left panel) and name them `QDRANT_HOST` and `QDRANT_API_KEY` respectively.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

QDRANT_HOST = os.getenv("QDRANT_HOST")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

if not QDRANT_HOST or not QDRANT_API_KEY:
    raise ValueError(
        "Missing one or more required environment variables: "
        "QDRANT_HOST, QDRANT_API_KEY."
    )

print("Qdrant credentials loaded successfully.")

In [ ]:
# Initialise an embedding function



#### **2.1.2** <font color=red> [5 marks] </font>
Load the embeddings to a vector database.

Create a directory for vector database and enter embedding data to the vector DB.

In [ ]:
import qdrant_client
from langchain_qdrant import QdrantVectorStore

# Initialize Qdrant Client
client = qdrant_client.QdrantClient(
    url=QDRANT_HOST,
    api_key=QDRANT_API_KEY
)

collection_name = "legal_docs"

# Check if collection exists
collections = client.get_collections().collections
collection_exists = any(c.name == collection_name for c in collections)

if not collection_exists:
    print(f"Collection '{collection_name}' not found in VectorDB so, creating and populating...")
    vector_store = QdrantVectorStore.from_texts(
        texts=chunks,
        embedding=embedding_function,
        url=QDRANT_HOST,
        api_key=QDRANT_API_KEY,
        collection_name=collection_name
    )
    print(f"Added {len(chunks)} chunks to Qdrant.")
else:
    print(f"Collection '{collection_name}' already exists. Connecting to existing store.")
    # Fixed: Added embedding parameter to allow dense retrieval
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embedding_function
    )

### **2.2 Create RAG Chain** <font color=red> [8 marks] </font><br>

#### **2.2.1** <font color=red> [5 marks] </font>
Form the complete RAG pipeline.

You can either create a chain or directly the pipeline

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama

try:
    # 2. Initialize the LLM
    llm = ChatOllama(model="llama3.2:3b", temperature=0)

    # 3. Set up retriever
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # 4. Prompt Template
    template = """You are an expert legal assistant. Answer the user's question using only the provided context below.
If you do not know the answer or if it's not explicitly mentioned in the context, say that you don't know. Do not make things up.

Context:
{context}

Question:
{question}

Answer:"""

    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # 5. Create the RAG chain
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("RAG chain successfully re-initialized.")

except ImportError as e:
    print(f"\nIMPORT ERROR: {e}")
    print("\n*** MANDATORY ACTION ***")
    print("1. Go to the menu: Runtime -> Restart session")
    print("2. Run THIS cell again immediately after restarting.")
except Exception as e:
    print(f"An error occurred: {e}")
    print("Ensure that 'vector_store' is defined in previous cells.")

#### **2.2.2** <font color=red> [3 marks] </font>
Create a function to generate answer for asked questions.

Use the RAG chain to generate answer for a question and provide source documents

In [ ]:
# Create a function for question answering
question = "What are the termination obligations under the NDA?"
response = rag_chain.invoke(question)

print(response)



In [ ]:
# Example question
question ="Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?"
response = rag_chain.invoke(question)
print(response)


## **3. RAG Evaluation** <font color=red> [10 marks] </font><br>

### **3.1 Evaluation and Inference** <font color=red> [10 marks] </font><br>

#### **3.1.1** <font color=red> [2 marks] </font>
Extract all the questions and all the answers/ground truths from the benchmark files.

Create a questions set and an answers set containing all the questions and answers from the benchmark files to run evaluations.

In [ ]:
# Create a question set by taking all the questions from the benchmark data
# Also create a ground truth/answer set
import json
import os
import random

benchmark_dir = "./benchmarks"  # adjust if your path differs
benchmark_files = ["contractnli.json", "cuad.json", "maud.json"]

all_questions = []
all_answers = []

for filename in benchmark_files:
    filepath = os.path.join(benchmark_dir, filename)
    if not os.path.exists(filepath):
        print(f"Warning: {filepath} not found, skipping.")
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    for test in data.get("tests", []):
        query = test.get("query", "").strip()
        snippets = test.get("snippets", [])

        # Collect all answer texts for this question as a combined ground truth
        ground_truth_parts = [s.get("answer", "").strip() for s in snippets if s.get("answer", "").strip()]
        ground_truth = " ".join(ground_truth_parts)

        if query and ground_truth:
            all_questions.append(query)
            all_answers.append(ground_truth)

print(f"Total questions loaded: {len(all_questions)}")
print(f"Total ground truths loaded: {len(all_answers)}")
print(f"\nSample question: {all_questions[0]}")
print(f"Sample ground truth: {all_answers[0][:300]}...")


#### **3.1.2** <font color=red> [5 marks] </font>
Create a function to evaluate the generated answers and retrieved contexts.

Evaluate the responses with *Ragas*. Additionally check the retrieval quality using 2 retrieval-driven metrics.

In [ ]:
import sys
import types
import pandas as pd
from datasets import Dataset

# Compatibility shim for environments where VertexAI classes were removed from langchain_community
try:
    from langchain_community.chat_models.vertexai import ChatVertexAI  # noqa: F401
except ModuleNotFoundError:
    vertex_chat_mod = types.ModuleType("langchain_community.chat_models.vertexai")
    class ChatVertexAI:
        pass
    vertex_chat_mod.ChatVertexAI = ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = vertex_chat_mod

try:
    from langchain_community.llms import VertexAI  # noqa: F401
except Exception:
    import langchain_community.llms as _lc_llms
    class VertexAI:
        pass
    _lc_llms.VertexAI = VertexAI

from ragas import evaluate
from ragas.metrics import faithfulness, context_recall, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama

ragas_llm = LangchainLLMWrapper(ChatOllama(model="llama3.2:3b", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_function)


def evaluate_rag_pipeline(questions, ground_truths):
    """
    Evaluates the RAG pipeline using a small metric set to avoid embedding issues.
    """
    eval_data = {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": [],
    }

    print(f"Processing {len(questions)} samples...")

    for q, gt in zip(questions, ground_truths):
        docs = retriever.invoke(q)
        eval_data["question"].append(q)
        eval_data["answer"].append(rag_chain.invoke(q))
        eval_data["contexts"].append([d.page_content for d in docs])
        eval_data["ground_truth"].append(gt)

    ds = Dataset.from_dict(eval_data)
    results = evaluate(
        ds,
        metrics=[faithfulness, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    return results, eval_data

#### **3.1.3** <font color=red> [3 marks] </font>
Draw inferences by evaluating answers to questions.

To save time and computing power, you can just run the evaluation on 10 randomly sampled questions.

In [ ]:
import random

# Select 10 random samples to save time and credits
sample_indices = random.sample(range(len(all_questions)), 1)
sample_questions = [all_questions[i] for i in sample_indices]
sample_ground_truths = [all_answers[i] for i in sample_indices]

#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Run the evaluation using the function defined in 3.1.2
results, eval_data = evaluate_rag_pipeline(sample_questions, sample_ground_truths)

# Display results
print("\nEvaluation Results:")
display(results.to_pandas())

## **4. Conclusion** <font color=red> [5 marks] </font><br>

### **4.1 Conclusions and insights** <font color=red> [5 marks] </font><br>

#### **4.1.1** <font color=red> [5 marks] </font>
Conclude with the results here. Include the insights gained about the data, model pipeline, the RAG process and the results obtained.